----
# **<span style="color:darkmagenta">PROYECTO EDA - BANK MARKETING</span>**
# **<span style="color:darkmagenta">Limpieza de datos</span>**
----

---
---
## <span style="color:gray">**1. Importación de librerías**</span> 📂

In [1]:
# Tratamiento de datos
import numpy as np
import pandas as pd 
from IPython.display import display

# Visualización de datos
import seaborn as sns
import matplotlib.pyplot as plt

# Configuración de ruta
import sys
sys.path.append('../')

# Importación de funciones personalizadas
from src.soporte2_limpieza import *


---
---
## <span style="color:gray">**2. Carga de datos**</span> 📥

In [2]:
# Datos CSV
bank_data = pd.read_csv("../data/raw/bank-additional.csv")

# Datos Excel
customer_data = pd.read_excel("../data/raw/customer-details.xlsx", sheet_name=None, index_col=0) # Devuelve un diccionario con todas las hojas

In [3]:
# Unión de las hojas de excel en un solo DataFrame
customer_data = pd.concat(customer_data.values(), ignore_index=True)

In [4]:
# Configuración para mostrar todas las columnas
pd.set_option('display.max_columns', None)

---
---

----
# <span style="color:darkmagenta">**Desarrollo del proyecto - 2**</span>
----


---
---
## <span style="color:gray">**Limpieza de datos**</span> 🧹

Antes de continuar con el análisis exploratorio de datos (EDA), creamos copias de los DataFrames originales.
Esto nos permite manipular, limpiar o transformar los datos sin afectar los originales, asegurando que siempre podamos recuperar la información original si es necesario.

In [5]:
bank_data_limpio = bank_data.copy()
customer_data_limpio = customer_data.copy()

---
## `bank_data_limpio`

### <span style="color:darkgray">**1. Estandarización de nombres de columnas**</span>


Antes de realizar la estandarización de nombres de columnas, renombramos algunas columnas para asegurar nombres claros y consistentes. Esto permite evitar errores al llamar columnas, facilita la automatización de análisis y mantiene consistencia en todo el proyecto. Los pasos que vamos a seguir son:

1. Crear un diccionario con los nuevos nombres de las columnas

In [6]:
nuevos_nombres = {
    "marital": "marital_status",
    "education": "education_level",
    "default": "default",
    "housing": "housing_loan",
    "loan": "personal_loan",
    "contact": "contact_method",
    "duration": "call_duration_sec",
    "campaign": "campaign_contacts",
    "pdays": "days_since_last_contact",
    "previous": "previous_contacts",
    "poutcome": "previous_campaign_outcome",
    "emp.var.rate": "employment_variation_rate",
    "cons.price.idx": "consumer_price_index",
    "cons.conf.idx": "consumer_confidence_index",
    "euribor3m": "euribor_3m_rate",
    "nr.employed": "number_employees",
    "y": "subscribed",
    "date": "contact_date",
    "id_": "ID"
}

2. Aplicar el diccionario con rename

In [7]:
bank_data_limpio.rename(columns=nuevos_nombres, inplace=True)

3. Estandarización de nombres de las columnas

In [8]:
# Llamamos a la función para estandarizar nombres de columnas guardada en el archivo de soporte
estandar_columns(bank_data_limpio)

4. Comprobamos que se ha ejecutado correctamente

In [9]:
print(bank_data_limpio.columns)

Index(['unnamed:_0', 'age', 'job', 'marital_status', 'education_level',
       'default', 'housing_loan', 'personal_loan', 'contact_method',
       'call_duration_sec', 'campaign_contacts', 'days_since_last_contact',
       'previous_contacts', 'previous_campaign_outcome',
       'employment_variation_rate', 'consumer_price_index',
       'consumer_confidence_index', 'euribor_3m_rate', 'number_employees',
       'subscribed', 'contact_date', 'latitude', 'longitude', 'id'],
      dtype='str')


### <span style="color:darkgray">**2. Eliminación de columnas irrelevantes**</span>

- **Comprobamos si la columna `unnamed:_0` del DataFrame bank_data tiene nulos o duplicados**

In [10]:
print(f"El número de nulos de la columna 'unnamed:_0' es: {bank_data_limpio['unnamed:_0'].isnull().sum()}")

El número de nulos de la columna 'unnamed:_0' es: 0


In [11]:
print(f"El número de nulos de la columna 'unnamed:_0' es: {bank_data_limpio['unnamed:_0'].duplicated().sum()}")

El número de nulos de la columna 'unnamed:_0' es: 1968


Como tiene duplicados, no nos aporta información relevante, por lo tanto, la eliminamos.

- **Variable `call_duration_sec`**

Esta información solo se conoce después de la llamada, por lo que no estaría disponible en el momento de predecir si un cliente aceptará o no el depósito. Incluirla podría generar predicciones poco realistas, por lo tanto, se elimina.

- **Variables `latitude` y `longitude`**

Los registros de estas variables provienen de campañas realizadas por un banco en Portugal, por lo tanto, solo reflejan pequeñas diferencias geográficas dentro del mismo país. Además, no tienen una relación directa con la decisión del cliente de suscribir un depósito a plazo. En consecuencia, aportan poca información al modelo y por ello las eliminamos.

- **Variable `days_since_last_contact`**

Se observa que esta variable utiliza el valor 999 para indicar que el cliente nunca había sido contactado previamente. Para mejorar la interpretación de la variable, creamos una nueva variable binaria `contacted_before` que indica si el cliente fue contactado anteriormente (1) o no (0). Una vez creada, se puede eliminar la columna `days_since_last_contact`.

In [12]:
bank_data_limpio['contacted_before'] = (bank_data_limpio['days_since_last_contact'] != 999).astype(int)

**Eliminamos las columnas anteriores**

In [13]:
# Llamamos a la función para eliminar columnas de un DataFrame guardada en el archivo de soporte
bank_data_limpio = eliminar_columns(
    bank_data_limpio,
    ["unnamed:_0", "call_duration_sec", "latitude", "longitude", "days_since_last_contact"]
)

### <span style="color:darkgray">**3. Limpieza de strings**</span>

- **Transformamos los datos de las variables `marital_status`, `previous_campaign_outcome` y `id` a minúsculas**

In [14]:
# Llamamos a la función que transforma datos a minúscula guardada en el archivo de sorporte
minus(bank_data_limpio, ["marital_status", "previous_campaign_outcome", "id"])

- **Comprobamos los datos únicos de algunas variables y hacemos la limpieza de algunos de los datos**

In [15]:
datos_unicos(bank_data_limpio, ["job", "education_level", "consumer_price_index", "consumer_confidence_index", "euribor_3m_rate", "number_employees"])

Los datos únicos de la varible job son:

 <StringArray>
[    'housemaid',      'services',        'admin.',   'blue-collar',
    'technician',       'retired',    'management',    'unemployed',
 'self-employed',             nan,  'entrepreneur',       'student']
Length: 12, dtype: str

-----------------------------------------------------------------
Los datos únicos de la varible education_level son:

 <StringArray>
[           'basic.4y',         'high.school',            'basic.6y',
            'basic.9y', 'professional.course',                   nan,
   'university.degree',          'illiterate']
Length: 8, dtype: str

-----------------------------------------------------------------
Los datos únicos de la varible consumer_price_index son:

 <StringArray>
['93,994',      nan, '94,465', '93,918', '93,444', '93,798',   '93,2',
 '92,756', '92,843', '93,075', '92,893', '92,963', '92,469', '92,201',
 '92,379', '92,431', '92,649', '92,713', '93,369', '93,749', '93,876',
 '94,055', '94,21

- La variable `job` tiene los datos admin., blue-collar y self-employed, estandarizamos esos datos cambiando admin. por la palabra completa y, self-employed y blue-collar los separamos con "_".
- En la variable `education_level`, para mantener consistencia con el formato usado en la variable `job`, usamos el carácter "_" como separador de palabras.
- Las variables `consumer_price_index`, `consumer_confidence_index`, `euribor_3m_rate` y `number_employees` contienen valores decimales escritos con comas como separador decimal; para estandarizar el formato y poder convertirlos a números, reemplazamos las comas por puntos.

In [16]:
# Llamamos a la función reemplazar guardada en el archivo de soporte 
reemplazar(bank_data_limpio, ["job"], "admin.", "administrative")
reemplazar(bank_data_limpio, ["job"], "-", "_")
reemplazar(bank_data_limpio, ["education_level"], ".", "_")
reemplazar(bank_data_limpio, ["consumer_price_index", "consumer_confidence_index", "euribor_3m_rate", "number_employees"], ",", ".")

### <span style="color:darkgray">**4. Corrección de tipos de datos**</span>

- **Variable `age`**

Está en formato str, por lo tanto, la convertimos a numérico:

In [17]:
bank_data_limpio['age'] = bank_data_limpio['age'].astype('Int64')

- **Cambiamos las variables `default`, `housing_loan`, `personal_loan` y `contacted_before` a variables categóricas:** 

   - 1 = 'si'
   - 0 = 'no'

In [18]:
columnas = ["default", "housing_loan", "personal_loan", "contacted_before"]

for columna in columnas:
    bank_data_limpio[columna] = bank_data_limpio[columna].replace({1:'si', 0:'no'})

- **Transformamos `consumer_price_index`, `consumer_confidence_index` y `euribor_3m_rate` en variables numéricas**           

In [19]:
columnas = ["consumer_price_index", "consumer_confidence_index", "euribor_3m_rate"]

for columna in columnas:
    bank_data_limpio[columna] = pd.to_numeric(bank_data_limpio[columna], errors='coerce')

- **Variable `number_employees`**

Para convertir esta variable a números enteros, primero hay que pasar de string a float, hacer un redondeo al entero más cercano y finalmente se cambia a int:

In [20]:
bank_data_limpio['number_employees'] = bank_data_limpio['number_employees'].astype(float).round().astype(int)

- **Variable `contact_date`**

Para convertir la columna al formato datetime es necesario realizar un paso previo de limpieza, ya que los meses están escritos en español y python reconoce los nombres de los meses en inglés, por lo que no puede interpretar correctamente estos valores. Para ello seguimos los siguientes pasos:

1. Crear un diccionario para traducir los meses del español al inglés

In [21]:
meses = {
"enero":"January","febrero":"February","marzo":"March","abril":"April",
"mayo":"May","junio":"June","julio":"July","agosto":"August",
"septiembre":"September","octubre":"October","noviembre":"November","diciembre":"December"
}

2. Reemplazar los nombres de los meses en la columna de fechas usando el diccionario

In [22]:
bank_data_limpio["contact_date"] = bank_data_limpio["contact_date"].replace(meses, regex=True)

3. Convertir la columna de texto a formato fecha

In [23]:
bank_data_limpio["contact_date"] = pd.to_datetime(bank_data_limpio["contact_date"], format="%d-%B-%Y", errors="coerce")

Una vez convertida la variable a formato fecha, se crearán dos nuevas variables: una que contenga el año y otra el mes, con el objetivo de obtener información temporal más relevante para el análisis.

In [24]:
bank_data_limpio['year'] = bank_data_limpio['contact_date'].dt.year

In [25]:
bank_data_limpio['month'] = bank_data_limpio['contact_date'].dt.month_name()

- Comprobamos que se han creado correctamente:

In [26]:
bank_data_limpio.info()

<class 'pandas.DataFrame'>
RangeIndex: 43000 entries, 0 to 42999
Data columns (total 22 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   age                        37880 non-null  Int64         
 1   job                        42655 non-null  str           
 2   marital_status             42915 non-null  str           
 3   education_level            41193 non-null  str           
 4   default                    34019 non-null  object        
 5   housing_loan               41974 non-null  object        
 6   personal_loan              41974 non-null  object        
 7   contact_method             43000 non-null  str           
 8   campaign_contacts          43000 non-null  int64         
 9   previous_contacts          43000 non-null  int64         
 10  previous_campaign_outcome  43000 non-null  str           
 11  employment_variation_rate  43000 non-null  float64       
 12  consumer_price_

- Eliminamos la columna `contact_date`:

In [27]:
bank_data_limpio = eliminar_columns(bank_data_limpio, ["contact_date"])

### <span style="color:darkgray">**5. Tratamiento de valores nulos**</span>

Durante la exploración inicial del Dataset, identificamos columnas con valores nulos; a continuación, evaluaremos cómo tratarlos individualmente.

Antes de tratar los nulos generamos un DataFrame que contenga solo las columnas que tienen nulos:

In [28]:
df_nulos = bank_data_limpio.loc[:, bank_data_limpio.isnull().sum() > 0]
df_nulos.sample(3)

,age,job,marital_status,education_level,default,housing_loan,personal_loan,consumer_price_index,euribor_3m_rate,year,month
12027,39,administrative,single,high_school,no,no,si,94.465,4.947,2017.0,January
19181,31,administrative,married,university_degree,no,NaN,NaN,93.444,NaN,2018.0,May
41963,<NA>,housemaid,married,basic_6y,no,no,no,93.444,NaN,2015.0,January


Recordamos el porcentaje de nulos de cada columna:

In [29]:
(df_nulos.isnull().sum() / df_nulos.shape[0]*100).round(2)

age                     11.91
job                      0.80
marital_status           0.20
education_level          4.20
default                 20.89
housing_loan             2.39
personal_loan            2.39
consumer_price_index     1.10
euribor_3m_rate         21.53
year                     0.58
month                    0.58
dtype: float64

- **Variable `age`**

Esta variable es relevante para el análisis; por lo tanto, los valores faltantes se rellenan con la mediana de las edades, manteniendo la consistencia de la distribución y evitando la pérdida de información.

- **Variables `job`, `marital_status` y `education_level`**   

Estas variables son categóricas y contienen un bajo porcentaje de valores nulos. Para no perder información y poder incluir todas las filas en el análisis, se rellenan los nulos con la categoría 'unknown', indicando que el valor es desconocido.

- **Variable `default`**

Asumimos que la mayoría de los clientes no tienen impagos, por lo tanto, rellenamos los datos faltantes con 'no', manteniendo la consistencia de la columna binaria y evitando la pérdida de información

- **Variables `housing_loan` y `personal_loan`**

Estas variables tienen un porcentaje muy bajo de nulos. Para no perder información, rellenamos los valores faltantes con 'no', indicando que el cliente no tiene préstamos. Esto mantiene la consistencia de las columnas binarias y permite incluir todas las observaciones en el análisis.

- **Variable `consumer_price_index`**

Esta columna contiene valores numéricos y tiene pocos nulos (1.1 %). Para mantener la consistencia y no perder información, se rellenan los valores faltantes con la media de la columna, evitando alterar significativamente la distribución de los datos.

- **Variable `euribor_3m_rate`**

Aunque esta variable tiene un porcentaje relativamente alto de valores nulos, conviene conservarla porque es relevante: refleja las condiciones del mercado que pueden afectar la decisión de suscripción de los clientes. Para mantener la consistencia del dataset, los valores faltantes se rellenan con la media de la columna.

- **Variables `year` y `month`**

Estas variables tiene un porcentaje muy bajo de nulos. Para no perder información y mantener consistencia, los valores faltantes se rellenan con la moda, la cual representa el año y més más comunes de contacto en el dataframe.

Rellenamos los nulos como hemos comentado 

In [ ]:
# Variable age
bank_data_limpio["age"] = bank_data_limpio["age"].fillna(bank_data_limpio["age"].median())

# Llamamos a las funciones de rellenar nulos guardadas en el archivo de soporte
rellenar_nulos_cat(bank_data_limpio, ["job", "marital_status", "education_level"], "unknown")
rellenar_nulos_cat(bank_data_limpio, ["default", "housing_loan", "personal_loan"], "no")
rellenar_nulos_num(bank_data_limpio, ["consumer_price_index", "euribor_3m_rate"])
rellenar_nulos_moda(bank_data_limpio, ["year", "month"])

Comprobamos que se han eliminado todos los nulos correctemente:

In [31]:
bank_data_limpio.isnull().sum()

age                          0
job                          0
marital_status               0
education_level              0
default                      0
housing_loan                 0
personal_loan                0
contact_method               0
campaign_contacts            0
previous_contacts            0
previous_campaign_outcome    0
employment_variation_rate    0
consumer_price_index         0
consumer_confidence_index    0
euribor_3m_rate              0
number_employees             0
subscribed                   0
id                           0
contacted_before             0
year                         0
month                        0
dtype: int64

No se observan valores duplicados

---
## `customer_data_limpio`

Este Dataset está prácticamente limpio; por lo que únicamente se renombrarán algunas columnas para mantener consistencia con el dataset anterior y se convertirán los datos de la columna id a minúsculas, asegurando así que la unión de ambos datasets se realice correctamente.

### <span style="color:darkgray">**1. Estandarización de nombres de columnas**</span>

- Renombramos algunas columnas

In [32]:
nombres_nuevos = {
    "Kidhome": "num_kids_home",
    "Teenhome":	"num_teens_home",
    "Dt_Customer": "customer_since", 
    "NumWebVisitsMonth": "num_web_visits_month"
}

customer_data_limpio.rename(columns=nombres_nuevos, inplace=True)

- Estandarización de nombres de las columnas

In [33]:
estandar_columns(customer_data_limpio)

### <span style="color:darkgray">**2. Transformación de los datos de la variable `id` a minúsculas**</span>

In [34]:
minus(customer_data_limpio, ['id'])

---
---
## <span style="color:gray">**Validación final de los datasets**</span> ✅

- **Comprobamos que la limpieza de ambos dataset se ha realizado correctamente**


### **`bank_data_limpio`** 

In [35]:
# Llamamos a la función analisis_rapido guardada en la carpeta src
analisis_rapido(bank_data_limpio)



Las 3 primeras filas del Dataframe son:



,age,job,marital_status,education_level,default,housing_loan,personal_loan,contact_method,campaign_contacts,previous_contacts,previous_campaign_outcome,employment_variation_rate,consumer_price_index,consumer_confidence_index,euribor_3m_rate,number_employees,subscribed,id,contacted_before,year,month
0,38,housemaid,married,basic_4y,no,no,no,telephone,1,0,nonexistent,1.1,93.994,-36.4,4.857000,5191,no,089b39d8-e4d0-461b-87d4-814d71e0e079,no,2019.0,August
1,57,services,married,high_school,no,no,no,telephone,1,0,nonexistent,1.1,93.994,-36.4,3.616521,5191,no,e9d37224-cb6f-4942-98d7-46672963d097,no,2016.0,September
2,37,services,married,high_school,no,si,no,telephone,1,0,nonexistent,1.1,93.994,-36.4,4.857000,5191,no,3f9f49b5-e410-4948-bf6e-f9244f04918b,no,2019.0,February



-----------------------------------------------------------

La información básica del Dataframe es la siguiente:

<class 'pandas.DataFrame'>
RangeIndex: 43000 entries, 0 to 42999
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   age                        43000 non-null  Int64  
 1   job                        43000 non-null  str    
 2   marital_status             43000 non-null  str    
 3   education_level            43000 non-null  str    
 4   default                    43000 non-null  object 
 5   housing_loan               43000 non-null  object 
 6   personal_loan              43000 non-null  object 
 7   contact_method             43000 non-null  str    
 8   campaign_contacts          43000 non-null  int64  
 9   previous_contacts          43000 non-null  int64  
 10  previous_campaign_outcome  43000 non-null  str    
 11  employment_variation_rate  43000 non-null  float64
 1

age                          0
job                          0
marital_status               0
education_level              0
default                      0
housing_loan                 0
personal_loan                0
contact_method               0
campaign_contacts            0
previous_contacts            0
previous_campaign_outcome    0
employment_variation_rate    0
consumer_price_index         0
consumer_confidence_index    0
euribor_3m_rate              0
number_employees             0
subscribed                   0
id                           0
contacted_before             0
year                         0
month                        0
dtype: int64

Observamos que la variable `year` es de tipo float, lo cual no tiene sentido en este contexto, por lo que la convertimos a tipo int:

In [36]:
bank_data_limpio['year'] = bank_data_limpio['year'].astype('int')

Y los datos de la variable `month` empiezan por mayúsculas, los cambiamos a minúsculas para mantener la consistencia en todo el DataFrame:

In [37]:
minus(bank_data_limpio, ['month'])

### **`customer_data_limpio`** 

In [38]:
# Llamamos a la función analisis_rapido guardada en la carpeta src
analisis_rapido(customer_data_limpio)



Las 3 primeras filas del Dataframe son:



,income,num_kids_home,num_teens_home,customer_since,num_web_visits_month,id
0,161770,1,0,2012-04-04,29,089b39d8-e4d0-461b-87d4-814d71e0e079
1,85477,1,1,2012-12-30,7,e9d37224-cb6f-4942-98d7-46672963d097
2,147233,1,1,2012-02-02,5,3f9f49b5-e410-4948-bf6e-f9244f04918b



-----------------------------------------------------------

La información básica del Dataframe es la siguiente:

<class 'pandas.DataFrame'>
RangeIndex: 43170 entries, 0 to 43169
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   income                43170 non-null  int64         
 1   num_kids_home         43170 non-null  int64         
 2   num_teens_home        43170 non-null  int64         
 3   customer_since        43170 non-null  datetime64[us]
 4   num_web_visits_month  43170 non-null  int64         
 5   id                    43170 non-null  str           
dtypes: datetime64[us](1), int64(4), str(1)
memory usage: 2.0 MB

-----------------------------------------------------------

El número de nulos por columna del Dataframe es:



income                  0
num_kids_home           0
num_teens_home          0
customer_since          0
num_web_visits_month    0
id                      0
dtype: int64

Hemos verificado que la limpieza de ambos dataframes se ha realizado correctamente: 

- Los nombres de las columnas son claros y descriptivos.
- Los datos son consistentes entre sí.
- No se observan valores nulos.
- Las variables presentan formatos adecuados para análisis y modelado posteriores .

Además, las transformaciones aplicadas (como estandarización de texto, conversión de tipos numéricos y codificación de variables categóricas) aseguran que ambos dataframes puedan combinarse y utilizarse de manera confiable.

- **Guardamos los dataset limpios**

In [39]:
bank_data_limpio.to_csv("../data/output/bank_data_limpio.csv", index=False)
customer_data_limpio.to_csv("../data/output/customer_data_limpio.csv", index=False)

---